# UNet

The UNet is our $\varepsilon_\theta(x_t, t)$ — it takes a noisy image and a timestep and predicts the noise.

Architecture:
- **Encoder**: series of blocks that downsample, each increasing channel depth
- **Bottleneck**: deepest representation, no spatial change
- **Decoder**: series of blocks that upsample, each receiving the corresponding encoder skip
- **Time embedding**: sinusoidal encoding of t, projected and injected into every block

In [1]:
import torch
import torch.nn as nn
import math

## Time Embedding

Sinusoidal encoding maps scalar $t$ to a vector, same formula as transformer positional encodings:

$$\text{emb}_{2i} = \sin\left(\frac{t}{10000^{2i/d}}\right), \quad \text{emb}_{2i+1} = \cos\left(\frac{t}{10000^{2i/d}}\right)$$

This is fixed. A small MLP then projects it to match each block's channel dimension.

In [2]:
def sinusoidal_embedding(t, dim):
    # t: (B,) integer timesteps, dim: embedding size
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)  # (B, dim)

## Basic Block

Each block: two convolutions with GroupNorm and ReLU, plus time injection.

Time is added after the first conv — projected to match channel depth, then broadcast spatially.

In [5]:
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act   = nn.ReLU()
        self.time_proj = nn.Linear(time_dim, out_ch)

    def forward(self, x, t_emb):
        x = self.act(self.norm1(self.conv1(x)))
        x = x + self.time_proj(t_emb)[:, :, None, None]  # broadcast over H, W
        x = self.act(self.norm2(self.conv2(x)))
        return x

## UNet

The encoder path halves spatial resolution at each stage while doubling channels, building an increasingly abstract representation. The decoder path reverses this, with skip connections from the corresponding encoder stage concatenated at each level — these carry fine spatial detail that the bottleneck representation has lost. The final 1×1 convolution projects back to the input channel count.

In [10]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=32, time_dim=128, n_labels=10):
        super().__init__()
        self.time_dim = time_dim
        self.n_labels = n_labels
        
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.ReLU(),
            nn.Linear(time_dim * 4, time_dim),
        )
        
        self.label_emb = nn.Embedding(n_labels, time_dim)

        # encoder
        self.enc1 = Block(in_ch,      base_ch,     time_dim)
        self.enc2 = Block(base_ch,    base_ch * 2, time_dim)
        self.enc3 = Block(base_ch*2,  base_ch * 4, time_dim)
        self.down  = nn.MaxPool2d(2)

        # bottleneck
        self.bot   = Block(base_ch*4, base_ch * 4, time_dim)

        # decoder — in_ch doubles because of skip concatenation
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3  = Block(base_ch*4 + base_ch*4, base_ch * 2, time_dim)
        self.dec2  = Block(base_ch*2 + base_ch*2, base_ch,     time_dim)
        self.dec1  = Block(base_ch   + base_ch,   base_ch,     time_dim)

        self.out   = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t, label):
        # pad to nearest multiple of 8 so downsampling/upsampling is symmetric
        H, W = x.shape[2], x.shape[3]
        pad_h = (8 - H % 8) % 8
        pad_w = (8 - W % 8) % 8
        x = torch.nn.functional.pad(x, (0, pad_w, 0, pad_h))

        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))
        l_emb = self.label_emb(label)
        t_emb += l_emb

        # encoder
        s1 = self.enc1(x,             t_emb)
        s2 = self.enc2(self.down(s1), t_emb)
        s3 = self.enc3(self.down(s2), t_emb)

        # bottleneck
        b  = self.bot(self.down(s3),  t_emb)

        # decoder — upsample then concatenate skip
        x  = self.dec3(torch.cat([self.up(b),  s3], dim=1), t_emb)
        x  = self.dec2(torch.cat([self.up(x),  s2], dim=1), t_emb)
        x  = self.dec1(torch.cat([self.up(x),  s1], dim=1), t_emb)

        x  = self.out(x)
        return x[:, :, :H, :W]  # crop back to original size

## Sanity Check

Input and output should have the same shape — the UNet predicts noise with the same dimensions as the image.

In [11]:
model = UNet(in_ch=3)
x = torch.randn(4, 3, 32, 32)
t = torch.randint(0, 1000, (4,))
labels = torch.randint(0, 10, (4,))

out = model(x, t, labels)
print(f'input:  {x.shape}')   # expect (4, 3, 32, 32)
print(f'output: {out.shape}') # expect (4, 3, 32, 32)